In [10]:
import pandas as pd  # 데이터프레임 처리

data = pd.DataFrame({  # 영화 줄거리와 다중 장르 라벨을 가진 데이터 생성
    'plot': [
        "A man fights crime in a futuristic city.",        # 영화 1 줄거리
        "A love story set in wartime.",                    # 영화 2 줄거리
        "Aliens invade Earth and a war begins.",           # 영화 3 줄거리
        "A detective solves a complicated crime case.",    # 영화 4 줄거리
        "A dramatic romance in the midst of a tragedy."    # 영화 5 줄거리
    ],
    'genres': [
        ['Action', 'Sci-Fi'],              # 영화 1 장르(다중 라벨)
        ['Romance', 'Drama'],              # 영화 2 장르(다중 라벨)
        ['Action', 'Sci-Fi', 'War'],       # 영화 3 장르(다중 라벨)
        ['Crime', 'Mystery'],              # 영화 4 장르(다중 라벨)
        ['Drama', 'Romance']               # 영화 5 장르(다중 라벨)
    ]
})
data  # 데이터 확인

,plot,genres
0,A man fights crime in a futuristic city.,"[Action, Sci-Fi]"
1,A love story set in wartime.,"[Romance, Drama]"
2,Aliens invade Earth and a war begins.,"[Action, Sci-Fi, War]"
3,A detective solves a complicated crime case.,"[Crime, Mystery]"
4,A dramatic romance in the midst of a tragedy.,"[Drama, Romance]"


In [11]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   plot    5 non-null      str   
 1   genres  5 non-null      object
dtypes: object(1), str(1)
memory usage: 406.0+ bytes


In [12]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(data['genres'])
print(y)
print(mlb.classes_)

label_df = pd.DataFrame(y, columns=mlb.classes_, index=data['plot'])
label_df

[[1 0 0 0 0 1 0]
 [0 0 1 0 1 0 0]
 [1 0 0 0 0 1 1]
 [0 1 0 1 0 0 0]
 [0 0 1 0 1 0 0]]
['Action' 'Crime' 'Drama' 'Mystery' 'Romance' 'Sci-Fi' 'War']


,Action,Crime,Drama,Mystery,Romance,Sci-Fi,War
plot,,,,,,,
A man fights crime in a futuristic city.,1,0,0,0,0,1,0
A love story set in wartime.,0,0,1,0,1,0,0
Aliens invade Earth and a war begins.,1,0,0,0,0,1,1
A detective solves a complicated crime case.,0,1,0,1,0,0,0
A dramatic romance in the midst of a tragedy.,0,0,1,0,1,0,0


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(data['plot'])

input_df = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=data['plot']
)
input_df

,aliens,and,begins,case,city,complicated,crime,detective,dramatic,earth,...,midst,of,romance,set,solves,story,the,tragedy,war,wartime
plot,,,,,,,,,,,,,,,,,,,,,
A man fights crime in a futuristic city.,0.000000,0.000000,0.000000,0.000000,0.442832,0.000000,0.357274,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
A love story set in wartime.,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.474125,0.000000,0.474125,0.000000,0.000000,0.000000,0.474125
Aliens invade Earth and a war begins.,0.408248,0.408248,0.408248,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.408248,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.408248,0.000000
A detective solves a complicated crime case.,0.000000,0.000000,0.000000,0.463693,0.000000,0.463693,0.374105,0.463693,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.463693,0.000000,0.000000,0.000000,0.000000,0.000000
A dramatic romance in the midst of a tragedy.,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.393795,0.000000,...,0.393795,0.393795,0.393795,0.000000,0.000000,0.000000,0.393795,0.393795,0.000000,0.000000


In [19]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

clf = OneVsRestClassifier(LogisticRegression())
clf.fit(X, y)

print('Done')

Done


In [21]:
test_plot = ['An alien spaceship lends in the middle of a war.']

X_test = vectorizer.transform(test_plot)
y_pred = clf.predict(X_test)
print(y_pred)

y_pred_proba = clf.predict_proba(X_test)
print(y_pred_proba)

y_pred = (y_pred_proba >= 0.3).astype(int)
print(y_pred)

y_pred_label = mlb.inverse_transform(y_pred)
y_pred_label

[[0 0 0 0 0 0 0]]
[[0.38623748 0.17121135 0.44411123 0.17121135 0.44411123 0.38623748
  0.20204508]]
[[1 0 1 0 1 1 0]]


[('Action', 'Drama', 'Romance', 'Sci-Fi')]

In [22]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import torch

tokenizer = Tokenizer(oov_token='OOV')
tokenizer.fit_on_texts(data['plot'])
X = tokenizer.texts_to_sequences(data['plot'])
X = pad_sequences(X, maxlen=10)
X = torch.tensor(X, dtype=torch.long)
X

tensor([[ 0,  0,  2,  5,  6,  4,  3,  2,  7,  8],
        [ 0,  0,  0,  0,  2,  9, 10, 11,  3, 12],
        [ 0,  0,  0, 13, 14, 15, 16,  2, 17, 18],
        [ 0,  0,  0,  2, 19, 20,  2, 21,  4, 22],
        [ 0,  2, 23, 24,  3, 25, 26, 27,  2, 28]])

In [23]:
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(data['genres'])
y = torch.tensor(y, dtype=torch.float)
y

tensor([[1., 0., 0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 1., 1.],
        [0., 1., 0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 1., 0., 0.]])

In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class MultiLabelNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, hidden = self.gru(x)
        output = self.fc(hidden[-1])
        return output

vocab_size = len(tokenizer.word_index) + 1

model = MultiLabelNet(vocab_size, embedding_dim=100, hidden_dim=64, output_dim=len(mlb.classes_))
model

MultiLabelNet(
  (embedding): Embedding(29, 100, padding_idx=0)
  (gru): GRU(100, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=7, bias=True)
)

In [25]:
from tqdm.auto import tqdm

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100

for epoch in tqdm(range(epochs)):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs} Loss: {loss.item()}')

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 10/100 Loss: 0.48450371623039246
Epoch 20/100 Loss: 0.31695377826690674
Epoch 30/100 Loss: 0.20809616148471832
Epoch 40/100 Loss: 0.14350296556949615
Epoch 50/100 Loss: 0.10526720434427261
Epoch 60/100 Loss: 0.08144263178110123
Epoch 70/100 Loss: 0.06569255143404007
Epoch 80/100 Loss: 0.054648008197546005
Epoch 90/100 Loss: 0.04651320353150368
Epoch 100/100 Loss: 0.04028270021080971


In [26]:
test_plot = ['An alien spaceship lands in the middle of a war.']

X_test = tokenizer.texts_to_sequences(test_plot)
X_test = pad_sequences(X_test, maxlen=10)
X_test = torch.tensor(X_test, dtype=torch.long)

model.eval()
with torch.no_grad():
    output = model(X_test)
    p = torch.sigmoid(output)
    pred = (p >= 0.5).int()
    pred_label = mlb.inverse_transform(pred)
    print(pred_label)

[('Action', 'Sci-Fi')]


In [27]:
!pip install transformers huggingface_hub

In [28]:
from transformers import BertTokenizer, BertModel

model_name = 'bert-base-uncased'
bert_tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = BertModel.from_pretrained(model_name)
bert_model

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

c:\Users\playdata2\NLP\nlp_venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\playdata2\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [29]:
import torch

def get_bert_embedding(plots):
    encoded = bert_tokenizer(plots, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        output = bert_model(**encoded)
    return output.last_hidden_state

plots = data['plot'].values.tolist()
X_tensor = get_bert_embedding(plots)
print(X_tensor.shape)

torch.Size([5, 12, 768])


In [30]:
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(data['genres'])
y_tensor = torch.tensor(y, dtype=torch.float)
y_tensor

tensor([[1., 0., 0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 1., 1.],
        [0., 1., 0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 1., 0., 0.]])

In [31]:
class MultiLabelNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        _, hidden = self.gru(x)
        output = self.fc(hidden[-1])
        return output

vocab_size = len(tokenizer.word_index) + 1

input_dim = X_tensor.shape[-1]
hidden_dim = 64
output_dim = y_tensor.shape[-1]

model = MultiLabelNet(input_dim, hidden_dim, output_dim)
model

MultiLabelNet(
  (gru): GRU(768, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=7, bias=True)
)

In [32]:
from tqdm.auto import tqdm

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100

for epoch in tqdm(range(epochs)):
    optimizer.zero_grad()
    output = model(X_tensor)
    loss = criterion(output, y_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs} Loss: {loss.item()}')

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 10/100 Loss: 0.3371884822845459
Epoch 20/100 Loss: 0.19859826564788818
Epoch 30/100 Loss: 0.13333922624588013
Epoch 40/100 Loss: 0.09823843836784363
Epoch 50/100 Loss: 0.07662402093410492
Epoch 60/100 Loss: 0.06203775480389595
Epoch 70/100 Loss: 0.05158939212560654
Epoch 80/100 Loss: 0.043814703822135925
Epoch 90/100 Loss: 0.0378546305000782
Epoch 100/100 Loss: 0.033167049288749695


In [ ]:
X_tensor = get_bert_embedding(test_plot)

with torch.no_grad():
    output = model(X_tensor)
    p = torch.sigmoid(output)
    pred = (p >= 0.7).int()
    pred_label = mlb.inverse_transform(pred)
    print(p)
    print(pred_label)

tensor([[0.7598, 0.0179, 0.2736, 0.0161, 0.2942, 0.8070, 0.1160]])
[('Action', 'Sci-Fi')]


In [40]:
test_plot = [
    "An alien spaceship lands in the middle of a war.",        # Action, Sci-Fi, War 예상
    "A young couple falls in love during a tragic event.",     # Romance, Drama 예상
    "A detective investigates a mysterious murder case."       # Crime, Mystery 예상
]

X_tensor = get_bert_embedding(test_plot)

with torch.no_grad():
    output = model(X_tensor)
    p = torch.sigmoid(output)
    pred = (p >= 0.5).int()
    pred_label = mlb.inverse_transform(pred)
    print(p)
    print(pred_label)

tensor([[0.7598, 0.0179, 0.2736, 0.0161, 0.2942, 0.8070, 0.1160],
        [0.5140, 0.0258, 0.4328, 0.0303, 0.5221, 0.5782, 0.0382],
        [0.0476, 0.9454, 0.0509, 0.9582, 0.0509, 0.0438, 0.0477]])
[('Action', 'Sci-Fi'), ('Action', 'Romance', 'Sci-Fi'), ('Crime', 'Mystery')]
